In [30]:
import matplotlib.pyplot as plt
import numpy as np
import random
import time

## Definición del entorno (laberinto 3x3)

In [31]:
n_rows, n_cols = 3, 3
actions = ["↑", "↓", "←", "→"]
n_actions = len(actions)
start_state = (0, 0)
goal_state = (2, 2)
reward_goal = 10
reward_step = -1

## Inicializar la tabla Q y funciones auxiliares

In [32]:
# Inicialización de la Q-table
Q = np.zeros((n_rows, n_cols, n_actions))

# Función para moverse dentro del laberinto
def next_state(state, action):
    r, c = state
    if action == 0 and r > 0:          # ↑
        r -= 1
    elif action == 1 and r < n_rows-1: # ↓
        r += 1
    elif action == 2 and c > 0:        # ←
        c -= 1
    elif action == 3 and c < n_cols-1: # →
        c += 1
    return (r, c)

def print_Q(Q):
    """Imprime la Q-table en formato legible"""
    for r in range(n_rows):
        for c in range(n_cols):
            print(f"Estado ({r},{c}): ", end="")
            for a in range(n_actions):
                print(f"{actions[a]}={Q[r,c,a]:5.2f} ", end="")
            print()
        print()


## Configurar parámetros de aprendizaje

In [33]:
alpha = 0.5      # tasa de aprendizaje
gamma = 0.9      # factor de descuento
epsilon = 0.3    # probabilidad de explorar
episodes = 20    # pocos episodios para ver la evolución gradual

In [34]:
# Inicializamos la matriz de recompensas
rewards = np.full((n_rows, n_cols), -1.0)  # por defecto -1

# Recompensa de la meta
rewards[2, 2] = 10.0

# Obstáculo peligroso
rewards[1, 0] = -10.0


# Mostrar la matriz
print("Matriz de recompensas:")
print(rewards)


Matriz de recompensas:
[[ -1.  -1.  -1.]
 [-10.  -1.  -1.]
 [ -1.  -1.  10.]]


## Entrenamiento

In [35]:
for ep in range(episodes):
    state = (0, 0)
    done = False
    total_reward = 0
    
    while not done:
        # Política ε-greedy
        if random.random() < epsilon:
            action = random.randint(0, n_actions-1)
        else:
            action = np.argmax(Q[state])

        new_state = next_state(state, action)
        reward = rewards[new_state]

        # Actualización Q-learning
        Q[state + (action,)] += alpha * (
            reward + gamma * np.max(Q[new_state]) - Q[state + (action,)]
        )

        state = new_state
        total_reward += reward

        if state == goal_state:
            done = True

    # Mostrar cada cierto número de episodios
    if (ep+1) % 5 == 0:
        print(f"\n Episodio {ep+1} completado — Recompensa total: {total_reward}")
        print_Q(Q)
        time.sleep(0.5)


 Episodio 5 completado — Recompensa total: 3.0
Estado (0,0): ↑=-1.21 ↓=-5.00 ←=-1.58 →=-0.19 
Estado (0,1): ↑=-0.14 ↓= 3.17 ←=-0.89 →=-0.88 
Estado (0,2): ↑=-0.50 ↓=-0.75 ←= 0.11 →=-0.50 

Estado (1,0): ↑=-0.50 ↓=-0.50 ←=-5.00 →= 1.86 
Estado (1,1): ↑= 0.00 ↓=-0.50 ←=-5.00 →= 6.75 
Estado (1,2): ↑=-0.50 ↓= 9.69 ←= 2.23 →= 0.00 

Estado (2,0): ↑=-5.00 ↓= 0.00 ←= 0.00 →= 0.00 
Estado (2,1): ↑= 1.64 ↓= 0.00 ←= 0.00 →= 0.00 
Estado (2,2): ↑= 0.00 ↓= 0.00 ←= 0.00 →= 0.00 


 Episodio 10 completado — Recompensa total: 7.0
Estado (0,0): ↑=-1.21 ↓=-6.77 ←=-1.27 →= 2.92 
Estado (0,1): ↑=-0.14 ↓= 5.79 ←=-0.89 →=-0.88 
Estado (0,2): ↑=-0.50 ↓=-0.75 ←= 0.11 →=-0.50 

Estado (1,0): ↑=-0.50 ↓=-0.50 ←=-5.00 →= 5.34 
Estado (1,1): ↑= 1.78 ↓=-0.01 ←=-5.38 →= 7.92 
Estado (1,2): ↑=-0.50 ↓= 9.99 ←= 2.23 →= 3.96 

Estado (2,0): ↑=-5.00 ↓= 0.00 ←= 0.00 →= 0.00 
Estado (2,1): ↑= 3.80 ↓= 0.24 ←= 0.00 →= 0.00 
Estado (2,2): ↑= 0.00 ↓= 0.00 ←= 0.00 →= 0.00 


 Episodio 15 completado — Recompensa total: 7.0
Es

## Derivar la política óptima final

In [36]:
policy = np.full((n_rows, n_cols), " ")
for r in range(n_rows):
    for c in range(n_cols):
        if (r, c) == goal_state:
            policy[r, c] = "🏁"
        else:
            policy[r, c] = actions[np.argmax(Q[r,c])]

print("Política aprendida después del entrenamiento:")
for row in policy:
    print(row)


Política aprendida después del entrenamiento:
['→' '↓' '←']
['→' '→' '↓']
['↓' '↑' '🏁']


## Ejemplo de recorrido siguiendo la política aprendida

In [37]:
state = (0,0)
path = [state]

while state != goal_state:
    action = np.argmax(Q[state])
    state = next_state(state, action)
    path.append(state)

print("Recorrido del agente según la política aprendida:")
print(" → ".join(str(p) for p in path))


Recorrido del agente según la política aprendida:
(0, 0) → (0, 1) → (1, 1) → (1, 2) → (2, 2)
